# EDA - Gold layer

Explorations that the dashboard was later built on (however with some slight changes) e.g. which events, countries and athletes dominate, how performance and speed are distributed and how participation has changed over time (more of that in transformations, also taken from Editor).

Each query is run with `.display()` so the result can be toggled to a chart directly in the notebook, a quick way to test which
visualisations are worth promoting to the dashboard before building them there.

In [0]:
# Row counts, sanity checks that everything built
# and the fact/dimension sizes make sense (one fact, smaller dimensions, a focused mart).

spark.sql("""
SELECT 'fct_results' AS table_name, COUNT(*) AS rows FROM marathos.gold.fct_results
UNION ALL SELECT 'dim_event',   COUNT(*) FROM marathos.gold.dim_event
UNION ALL SELECT 'dim_athlete', COUNT(*) FROM marathos.gold.dim_athlete
UNION ALL SELECT 'dim_date',    COUNT(*) FROM marathos.gold.dim_date
UNION ALL SELECT 'dim_country', COUNT(*) FROM marathos.gold.dim_country
UNION ALL SELECT 'mart_sweden', COUNT(*) FROM marathos.gold.mart_sweden
ORDER BY rows DESC
""").display()

## Event - distance vs length

Two marathon types that behave differently (distance races are timed, length races measure distance covered), so most views are split by `event_type`. 

Check the split and the average field size per type.


In [0]:
spark.sql("""
SELECT
  event_type,
  COUNT(*)                                  AS n_events,
  ROUND(AVG(event_number_of_finishers), 0)  AS avg_finishers
FROM marathos.gold.dim_event
GROUP BY event_type
ORDER BY n_events DESC
""").display()

## Big global events

Which events draw the most finishers across the whole dataset. A bar chart of this is a strong candidate for a global overview tile.


In [0]:
spark.sql("""
SELECT
  e.event_name,
  e.event_type,
  COUNT(*) AS n_finishes
FROM marathos.gold.fct_results f
JOIN marathos.gold.dim_event e ON f.event_id = e.event_id
GROUP BY e.event_name, e.event_type
ORDER BY n_finishes DESC
LIMIT 20
""").display()

## Country participation

Top countries by number of finishes + BONUS country dimension, for a map or a ranked bar chart.


In [0]:
spark.sql("""
SELECT
  c.country_name,
  c.continent,
  COUNT(*)                      AS n_finishes,
  COUNT(DISTINCT f.athlete_id)  AS n_athletes
FROM marathos.gold.fct_results f
JOIN marathos.gold.dim_athlete a ON f.athlete_id = a.athlete_id
JOIN marathos.gold.dim_country c ON a.athlete_country = c.country_code
GROUP BY c.country_name, c.continent
ORDER BY n_finishes DESC
LIMIT 20
""").display()

## Gender distribution

Overall gender split across all finishes. Skews heavily male, so a pie/donut here sets expectations for the Swedish view in the dashboard.

In [0]:
spark.sql("""
SELECT
  a.athlete_gender,
  COUNT(*) AS n_finishes
FROM marathos.gold.fct_results f
JOIN marathos.gold.dim_athlete a ON f.athlete_id = a.athlete_id
GROUP BY a.athlete_gender
ORDER BY n_finishes DESC
""").display()

## Speed - the recomputed metric

`recomputed_speed_kmh` was rebuilt from distance and time (the source
`athlete_average_speed` had impossible values). 

The range is now plausible per event type, then look at the distribution.

In [0]:
spark.sql("""
SELECT
  e.event_type,
  ROUND(AVG(f.recomputed_speed_kmh), 2) AS avg_speed_kmh,
  ROUND(MIN(f.recomputed_speed_kmh), 2) AS min_speed_kmh,
  ROUND(MAX(f.recomputed_speed_kmh), 2) AS max_speed_kmh
FROM marathos.gold.fct_results f
JOIN marathos.gold.dim_event e ON f.event_id = e.event_id
WHERE f.recomputed_speed_kmh IS NOT NULL
GROUP BY e.event_type
""").display()

## Participation over time

Finishes per year, using the BONUS date dimension. The sport is recent at scale even though the data is centuries old, a line chart shows the growth clearly.


In [0]:
spark.sql("""
SELECT
  d.year,
  COUNT(*) AS n_finishes
FROM marathos.gold.fct_results f
JOIN marathos.gold.dim_date d ON f.date_id = d.date_id
GROUP BY d.year
ORDER BY d.year
""").display()

## Age categories

Which age groups are most represented.

In [0]:
spark.sql("""
SELECT
  a.athlete_age_category,
  COUNT(*) AS n_finishes
FROM marathos.gold.fct_results f
JOIN marathos.gold.dim_athlete a ON f.athlete_id = a.athlete_id
GROUP BY a.athlete_age_category
ORDER BY n_finishes DESC
LIMIT 20
""").display()

## Sweden focus - minor changes aside, what the dashboard shows

`mart_sweden` -> to Swedish events (name ending in `(SWE)`), including the streamed LLM-generated Marathos events. These three queries mirror the dashboard datasets.

In [0]:
# Top Swedish events by finishes (dashboard top_events dataset)
# the dashboard uses a :top_events parameter; fixed to 10 here for the notebook
spark.sql("""
SELECT
  event_name,
  event_type,
  COUNT(*) AS n_finishes,
  ROUND(AVG(recomputed_speed_kmh), 2) AS avg_speed_kmh
FROM marathos.gold.mart_sweden
GROUP BY event_name, event_type
ORDER BY n_finishes DESC
LIMIT 10
""").display()

In [0]:
# Gender split by event type in Swedish events (dashboard gender_distribution dataset)
# pct computed with a window function - LLM suggestion
spark.sql("""
SELECT
  athlete_gender,
  event_type,
  COUNT(*) AS n_finishes,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct
FROM marathos.gold.mart_sweden
GROUP BY athlete_gender, event_type
""").display()


In [0]:
# Events by finishes and average speed (dashboard namn_products dataset)
# the dashboard uses a :top_events parameter
spark.sql("""
SELECT
  event_name,
  COUNT(*) AS n_finishes,
  ROUND(AVG(recomputed_speed_kmh), 2) AS avg_speed_kmh
FROM marathos.gold.mart_sweden
GROUP BY event_name
ORDER BY n_finishes DESC
LIMIT 10
""").display()